# ACT run C — 50 veri x4 + 500 simulati, mescolati dall'inizio

Veri: `caramelle-so101-50`, 4 copie con i metadati riparati (lo stesso
dataset nominato piu' volte nel merge si rompe): ~31% dei campioni, ogni
fotogramma vero visto ~2,6 volte (run B: 5,1). Simulati: output di
`act-caramelle-sim-gen`. `lerobot-train` come la run B, 100000 passi,
checkpoint ogni 25000. Il checkpoint si sceglie **sul braccio**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv; nproc; df -h /tmp | tail -1

In [ ]:
import glob, os, shutil
s = glob.glob('/kaggle/input/**/robotica/scripts/genera_act_sim.py', recursive=True)
assert s, 'pacchetto codice non trovato: e\' in Input?'
shutil.copytree(os.path.dirname(os.path.dirname(s[0])), '/tmp/robotica', dirs_exist_ok=True)
print('codice:', s[0])

In [ ]:
!pip install -q uv
!UV_CACHE_DIR=/tmp/uvcache uv venv -q -p 3.12 /tmp/venv
!UV_CACHE_DIR=/tmp/uvcache uv pip install -q --python /tmp/venv/bin/python --torch-backend=auto "lerobot[training,dataset]==0.6.1" "mujoco==3.3.7" scipy
!/tmp/venv/bin/python -c "import lerobot, mujoco, numpy; print('lerobot', lerobot.__version__, '| mujoco', mujoco.__version__, '| numpy', numpy.__version__)"

In [ ]:
import json, subprocess
sim = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/caramelle_sim/meta/info.json', recursive=True)]
assert len(sim) == 1, sim
es = json.load(open(sim[0] + '/meta/info.json'))['total_episodes']
SIM = sim[0]
print('simulati:', SIM, es, 'episodi')
if es != 500:
    print(f'ATTENZIONE: {es} simulati invece di 500')

In [ ]:
veri = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/meta/info.json', recursive=True)
        if json.load(open(p))['total_episodes'] == 50]
assert len(veri) == 1, veri
copie = [f'/tmp/veri_{x}' for x in 'abcd']
for copia in copie:
    shutil.copytree(veri[0], copia, copy_function=shutil.copyfile)  # /kaggle/input e' in sola lettura: niente permessi
    print(subprocess.run(['/tmp/venv/bin/python', '/tmp/robotica/scripts/ripara_meta_episodi.py', copia], capture_output=True, text=True).stdout)
ids = [f'so101/veri_{x}' for x in 'abcd'] + ['so101/sim']
r = subprocess.run(['/tmp/venv/bin/lerobot-edit-dataset', '--new_repo_id', 'so101/misto', '--new_root', '/tmp/dati',
                    '--operation.type', 'merge', '--operation.repo_ids', json.dumps(ids),
                    '--operation.roots', json.dumps(copie + [SIM])], capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
info = json.load(open('/tmp/dati/meta/info.json'))
fs = json.load(open(SIM + '/meta/info.json'))['total_frames']
print('episodi', info['total_episodes'], 'fotogrammi', info['total_frames'], f'-> veri {100 * (1 - fs / info["total_frames"]):.0f}% dei campioni')
assert info['total_episodes'] == 50 * 4 + es
for copia in copie:
    shutil.rmtree(copia)
REPO = 'so101/misto'

In [ ]:
!/tmp/venv/bin/lerobot-train \
    --dataset.repo_id=$REPO \
    --dataset.root=/tmp/dati \
    --dataset.image_transforms.enable=true \
    --policy.type=act \
    --policy.device=cuda \
    --policy.push_to_hub=false \
    --output_dir=/kaggle/working/act_caramelle \
    --job_name=act_run_c \
    --steps=100000 \
    --save_freq=25000 \
    --batch_size=8 \
    --wandb.enable=false

In [ ]:
base = '/kaggle/working/act_caramelle/checkpoints'
numeri = {int(n): n for n in os.listdir(base) if n.isdigit()}
print('checkpoint:', sorted(numeri))
for passi in sorted(numeri):
    uscita = shutil.make_archive(f'/kaggle/working/modello_act_{passi}', 'zip', os.path.join(base, numeri[passi], 'pretrained_model'))
    print('pronto:', uscita, os.path.getsize(uscita) / 1e6, 'MB')